In [ ]:
!pip install datasets
!pip install jiwer
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is 

In [ ]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
import evaluate

In [ ]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")


README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

librispeech_asr_adversarial.py:   0%|          | 0.00/5.59k [00:00<?, ?B/s]

The repository for RaphaelOlivier/librispeech_asr_adversarial contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/RaphaelOlivier/librispeech_asr_adversarial.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating natural split: 0 examples [00:00, ? examples/s]

Generating adv_0.04 split: 0 examples [00:00, ? examples/s]

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [ ]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from utils import transcribe_audio

In [ ]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
# Extract audio arrays, sampling rates, and ground truths
audio_arrays = [example["audio"]["array"] for example in librispeech]
sampling_rates = [example["audio"]["sampling_rate"] for example in librispeech]
ground_truths = [example["true_text"].lower().strip() for example in librispeech]

# Generate transcriptions for all samples
try:
    transcriptions = [
        transcribe_audio(audio_array, sampling_rate, processor, model).lower().strip()
        for audio_array, sampling_rate in zip(audio_arrays, sampling_rates)
    ]
except Exception as e:
    print(f"Error during batch transcription: {e}")
    transcriptions = []



In [ ]:
import evaluate

# load both metrics
wer_metric = evaluate.load("wer")    # Word‑Error‑Rate :contentReference[oaicite:0]{index=0}
cer_metric = evaluate.load("cer")


# compute them in one shot
avg_wer = wer_metric.compute(predictions=transcriptions, references=ground_truths)
avg_cer = cer_metric.compute(predictions=transcriptions, references=ground_truths)

print(f"Average WER: {avg_wer:.4f} ({avg_wer*100:.2f}%)")
print(f"Average CER: {avg_cer:.4f} ({avg_cer*100:.2f}%)")


Average WER: 0.0290 (2.90%)
Average CER: 0.0082 (0.82%)


In [ ]:
from pgd import pgd_attack

In [ ]:
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

example = librispeech[0]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]  # Ground truth transcription
target_transcription = "HELLO WORLD"  # Target transcription

# Run PGD attack
adversarial_waveform = pgd_attack(
    audio_array=audio_array,
    ground_truth=ground_truth,
    target_transcription=target_transcription,
    model=model,
    processor=processor,
    epsilon=0.05,
    alpha=0.005,
    num_iter=10
)

original_transcription = transcribe_audio(audio_array, 16000, processor, model)
adversarial_transcription = transcribe_audio(adversarial_waveform, 16000, processor, model)


# Calculate CER and WER
cer_original = cer_metric.compute(predictions=[original_transcription], references=[ground_truth])
wer_original = wer_metric.compute(predictions=[original_transcription], references=[ground_truth])
cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])

# Display audio
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
print("Adversarial Audio:")
display(ipd.Audio(adversarial_waveform, rate=16000))

# Print transcription and metrics
print(f"Ground Truth: {ground_truth}")
print(f"Original Transcription: {original_transcription}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"Original CER: {cer_original:.2f}")
print(f"Original WER: {wer_original:.2f}")
print(f"Adversarial CER: {cer:.2f}")
print(f"Adversarial WER: {wer:.2f}")

Original Audio:


Adversarial Audio:


Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Original Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGILY POSIV
Original CER: 0.00
Original WER: 0.00
Adversarial CER: 0.15
Adversarial WER: 0.25


In [15]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from datasets import load_dataset
import evaluate
import IPython.display as ipd
import numpy as np
from torch.amp import autocast
from tqdm.notebook import tqdm

# Initialize model, processor, and metrics
device = "cuda" if torch.cuda.is_available() else "cpu"


# Select three sample indices for demonstration
selected_indices = [0, 1, 2]
epsilon_values = [0.01, 0.02, 0.05, 0.1, 0.2]
alpha_values = [0.001, 0.002, 0.005, 0.01, 0.02]
target_transcription = "HELLO WORLD"
batch_size = 4

# Pre-cache original transcriptions for demo samples
demo_original_transcriptions = {}
for idx in selected_indices:
    audio_array = librispeech[idx]["audio"]["array"]
    demo_original_transcriptions[idx] = transcribe_audio([audio_array], 16000, processor, model)[0]

# Evaluate for each epsilon-alpha pair
for epsilon, alpha in zip(epsilon_values, alpha_values):
    print(f"\n=== Epsilon: {epsilon}, Alpha: {alpha} ===")
    cer_list = []
    wer_list = []
    demo_samples = []

    # Process dataset in batches with progress bar
    num_batches = (len(librispeech) + batch_size - 1) // batch_size
    for batch_start in tqdm(range(0, len(librispeech), batch_size), total=num_batches, desc="Processing Batches"):
        batch_end = min(batch_start + batch_size, len(librispeech))
        batch_indices = list(range(batch_start, batch_end))
        batch_audio = [librispeech[idx]["audio"]["array"] for idx in batch_indices]
        batch_ground_truth = [librispeech[idx]["true_text"] for idx in batch_indices]

        # Run PGD attack on batch
        batch_adversarial_waveforms = []
        for audio_array in batch_audio:
            with autocast('cuda'):
                adv_waveform = pgd_attack(
                    audio_array=audio_array,
                    ground_truth="",  # Unused in pgd_attack
                    target_transcription=target_transcription,
                    model=model,
                    processor=processor,
                    epsilon=epsilon,
                    alpha=alpha,
                    num_iter=10,
                    device=device
                )
            batch_adversarial_waveforms.append(adv_waveform)

        # Transcribe adversarial audio in batch
        adversarial_transcriptions = transcribe_audio(batch_adversarial_waveforms, 16000, processor, model)

        # Compute metrics
        for idx, adv_transcription, ground_truth in zip(batch_indices, adversarial_transcriptions, batch_ground_truth):
            cer = cer_metric.compute(predictions=[adv_transcription], references=[ground_truth])
            wer = wer_metric.compute(predictions=[adv_transcription], references=[ground_truth])
            cer_list.append(cer)
            wer_list.append(wer)

            # Store demo sample info
            if idx in selected_indices:
                demo_samples.append({
                    'original_audio': librispeech[idx]["audio"]["array"],
                    'adversarial_audio': batch_adversarial_waveforms[batch_indices.index(idx)],
                    'original_transcription': demo_original_transcriptions[idx],
                    'adversarial_transcription': adv_transcription,
                    'ground_truth': ground_truth,
                    'cer': cer,
                    'wer': wer
                })

    # Compute and print overall metrics
    overall_cer = np.mean(cer_list)
    overall_wer = np.mean(wer_list)
    print(f"Overall CER: {overall_cer:.2f}")
    print(f"Overall WER: {overall_wer:.2f}")

    # Display demo samples
    for i, sample in enumerate(demo_samples):
        print(f"\nSample {selected_indices[i]}:")
        print(f"Ground Truth: {sample['ground_truth']}")
        print(f"Original Transcription: {sample['original_transcription']}")
        print(f"Adversarial Transcription: {sample['adversarial_transcription']}")
        print(f"CER: {sample['cer']:.2f}")
        print(f"WER: {sample['wer']:.2f}")
        print("Original Audio:")
        display(ipd.Audio(sample['original_audio'], rate=16000))
        print("Adversarial Audio:")
        display(ipd.Audio(sample['adversarial_audio'], rate=16000))


=== Epsilon: 0.01, Alpha: 0.001 ===


Processing Batches:   0%|          | 0/22 [00:00<?, ?it/s]

KeyboardInterrupt: 